# ATLAS — FMNIST @107M · Stack v2 (3 configs)

**Decisão:** batch CPU cancelado. Seguir com **warmup+cosine** como receita base + testar **multi_scale** e **tri_scale** em @107M na GPU.

**Como usar:**
1. Runtime → **GPU (A100 ou H100)**
2. Se o repo for **privado**: Colab → 🔑 **Secrets** → `GITHUB_TOKEN` (PAT com scope `repo`)
3. **Run all** (~**35–50 min** total)

**3 experimentos × 3 seeds = 9 treinos:**

| ID | Stack | O que testa |
|----|-------|-----------|
| `exp-100m-wc` | **warmup+cosine** | Receita campeã CPU (89,93%) — confirmar @107M GPU |
| `exp-100m-ms` | multi_scale + LS + wc | Campeão CIFAR (+3,48 pp) |
| `exp-100m-tri` | tri_scale + LS + wc | Recorde CPU pequeno (92,98%) |

**Escala:** `width_mult=16`, `hidden_dim=2048`, ~107M params, 1000 steps, bf16.

**Referência CPU (não reroda):** warmup+cosine **89,93%** | baseline plain **86,12%**

**Branch:** `cursor/promissora-scale-c396` · repo `Revenn0/0111`

In [ ]:
# @title 1. GPU
import torch
assert torch.cuda.is_available(), "Ative GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("PyTorch:", torch.__version__)

In [ ]:
# @title 2. Clone repo ATLAS
import os, subprocess, sys, shutil

REPO_SLUG = "Revenn0/0111"
BRANCH = "cursor/promissora-scale-c396"
DIR = "0111"

def get_token():
    # 1) Colab Secrets: chave GITHUB_TOKEN
    try:
        from google.colab import userdata
        t = userdata.get("GITHUB_TOKEN")
        if t:
            return t.strip()
    except Exception:
        pass
    # 2) variável de ambiente
    t = os.environ.get("GITHUB_TOKEN", "").strip()
    if t:
        return t
    # 3) prompt (repo privado)
    try:
        from getpass import getpass
        t = getpass("Repo privado: cole um GitHub token (scope repo) ou Enter p/ tentar anônimo: ").strip()
        return t or None
    except Exception:
        return None

def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd) + (f"  (cwd={cwd})" if cwd else ""))
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    if check and r.returncode:
        raise subprocess.CalledProcessError(r.returncode, cmd, r.stdout, r.stderr)
    return r

def repo_url(token=None):
    if token:
        return f"https://{token}@github.com/{REPO_SLUG}.git"
    return f"https://github.com/{REPO_SLUG}.git"

def clone_or_update(token=None):
    url = repo_url(token)

    if os.path.isdir(DIR):
        try:
            run(["git", "remote", "set-url", "origin", url], cwd=DIR)
            run(["git", "fetch", "origin", BRANCH], cwd=DIR)
            run(["git", "checkout", BRANCH], cwd=DIR)
            run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=DIR)
            return
        except subprocess.CalledProcessError:
            print("Repo local inválido — removendo e reclonando…")
            shutil.rmtree(DIR, ignore_errors=True)

    try:
        run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, DIR])
        return
    except subprocess.CalledProcessError:
        shutil.rmtree(DIR, ignore_errors=True)

    # fallback: clone default branch + checkout
    run(["git", "clone", "--depth", "1", url, DIR])
    run(["git", "fetch", "origin", BRANCH], cwd=DIR)
    run(["git", "checkout", BRANCH], cwd=DIR)

token = get_token()
try:
    clone_or_update(token)
except subprocess.CalledProcessError:
    if not token:
        raise RuntimeError(
            "Clone falhou (exit 128). O repo parece PRIVADO.\n"
            "→ Colab: ícone 🔑 Secrets → GITHUB_TOKEN = seu Personal Access Token (scope repo)\n"
            "→ Ou rode de novo e cole o token quando pedir."
        ) from None
    raise

os.chdir(DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision"], check=False)
head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("OK:", os.getcwd(), "| branch:", BRANCH, "| commit:", head)

In [ ]:
# @title 3. Verificar escala ~107M params
import json, subprocess

out = subprocess.check_output([
    sys.executable, "-c",
    "from train_baseline import build_model, TrainConfig; "
    "c=TrainConfig(width_mult=16.0, hidden_dim=2048); "
    "n=sum(p.numel() for p in build_model(c).parameters()); print(n)"
], text=True).strip()
params_m = int(out) / 1e6
print(f"Params: {params_m:.2f}M (alvo ~107M)")
assert 100 <= params_m <= 115, f"Escala fora da faixa: {params_m:.1f}M"

In [ ]:
# @title 4. Rodar 3 configs @107M (GPU)
import json, subprocess, sys, time
from pathlib import Path
from datetime import datetime, timezone

SCALE = [
    "--width_mult", "16.0", "--hidden_dim", "2048",
    "--batch_size", "128", "--lr", "0.001", "--weight_decay", "1e-4",
    "--device", "cuda", "--amp", "--num_workers", "2",
    "--steps", "1000", "--eval_every", "250", "--log_every", "200",
]
WC = ["--warmup_steps", "100", "--scheduler", "cosine"]
LS = ["--label_smoothing", "0.1"]
RESULT_DIR = Path("results/100m_colab")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# (id, name, novelty, lesson, extra_args)
EXPERIMENTS = [
    ("exp-100m-wc", "warmup_cosine_100m", "RECOMB exp-007/011", "warmup+cosine (stack base)", WC),
    ("exp-100m-ms", "multi_scale_ls_wc_100m", "NOVEL exp-042/047", "multi_scale + LS + wc", ["--mixing", "multi_scale_blend"] + LS + WC),
    ("exp-100m-tri", "tri_scale_ls_wc_100m", "NOVEL exp-053", "tri_scale + LS + wc", ["--mixing", "tri_scale_blend"] + LS + WC),
]

CPU_BASELINE = 0.8612  # régua CPU já medida
CPU_LEADER = 0.8993    # exp-100m-002 warmup+cosine
LOG_ENTRIES = []
T0_TOTAL = time.time()

def run_one_seed(eid, extra, seed):
    out = RESULT_DIR / f"{eid}_seed{seed}.json"
    cmd = [sys.executable, "train_baseline.py"] + SCALE + extra + [
        "--seed", str(seed), "--result_path", str(out)
    ]
    subprocess.run(cmd, check=True)
    return json.loads(out.read_text())

for eid, name, nov, lesson, extra in EXPERIMENTS:
    print(f"\n{'='*60}\n{eid} | {lesson}\n{'='*60}")
    t0 = time.time()
    per_seed = []
    for seed in [1000, 1001, 1002]:
        r = run_one_seed(eid, extra, seed)
        per_seed.append(r)
        print(f"  seed {seed}: {r['best_val_acc']*100:.2f}%  ({r['steps_per_sec']:.1f} steps/s)")
    accs = [r["best_val_acc"] for r in per_seed]
    mean = sum(accs) / len(accs)
    std = (sum((a-mean)**2 for a in accs)/max(1,len(accs)-1))**0.5 if len(accs)>1 else 0
    entry = {
        "id": eid,
        "name": name,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "source": "colab_a100_100m",
        "seeds": [1000, 1001, 1002],
        "result": {
            "n_seeds": 3,
            "best_val_acc_mean": mean,
            "best_val_acc_std": std,
            "wall_time_s_mean": sum(r["wall_time_s"] for r in per_seed)/3,
            "steps_per_sec_mean": sum(r["steps_per_sec"] for r in per_seed)/3,
            "per_seed": per_seed,
        },
        "delta_vs_cpu_baseline_pp": round((mean - CPU_BASELINE)*100, 2),
        "delta_vs_cpu_leader_pp": round((mean - CPU_LEADER)*100, 2),
        "novelty_note": nov,
        "lesson": lesson,
    }
    LOG_ENTRIES.append(entry)
    print(f"  >> média {mean*100:.2f}% ± {std*100:.2f}%  Δleader={entry['delta_vs_cpu_leader_pp']:+.2f}pp  ({time.time()-t0:.0f}s)")

out_path = Path("experiments_log_100m_colab.jsonl")
with out_path.open("w") as f:
    for e in LOG_ENTRIES:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
elapsed = time.time() - T0_TOTAL
print(f"\nSalvo: {out_path.resolve()}")
print(f"Tempo total: {elapsed/60:.1f} min ({elapsed:.0f}s)")

In [ ]:
# @title 5. Ranking + veredito
CPU_WC = 0.8993
CPU_BASE = 0.8612

rows = [(e["id"], e["lesson"], e["result"]["best_val_acc_mean"], e["result"]["best_val_acc_std"], "GPU") for e in LOG_ENTRIES]
rows.sort(key=lambda x: -x[2])

print(f"{'Rank':<5} {'ID':<16} {'Acc':>8} {'±':>6} {'Δ wc':>7} {'Veredito':<14} {''}")
print("-"*72)
for i, (eid, label, acc, std, src) in enumerate(rows, 1):
    d_wc = (acc - CPU_WC) * 100
    if d_wc >= 0.8 and std <= 0.003:
        v = "REVOLUCIONÁRIA"
    elif d_wc >= 0.3:
        v = "PROMISSORA"
    elif d_wc >= -0.29:
        v = "INCONCLUSIVA"
    else:
        v = "REFUTADA"
    print(f"{i:<5} {eid:<16} {acc*100:7.2f}% {std*100:5.2f}% {d_wc:+6.2f}pp {v:<14} {label}")

best = max(LOG_ENTRIES, key=lambda e: e["result"]["best_val_acc_mean"])
b = best["result"]["best_val_acc_mean"]
print(f"\n🏆 Melhor: {best['id']} = {b*100:.2f}%")
if best["id"] in ("exp-100m-ms", "exp-100m-tri") and b > CPU_WC:
    print("→ Stack v2 confirmada: warmup+cosine + multi/tri_scale @107M")
elif best["id"] == "exp-100m-wc":
    print("→ warmup+cosine sozinho continua campeão; mixer não ajudou em 107M")
print(f"\nRef CPU: warmup+cosine {CPU_WC*100:.2f}% | baseline plain {CPU_BASE*100:.2f}%")

In [ ]:
# @title 6. Download resultados
from google.colab import files
files.download("experiments_log_100m_colab.jsonl")